# DPX SEC tips on the Hamilton STAR

Buffer exchange with DPX 1 mL size-exclusion tips (wide-bore 3.2 mm), using the independent 1000 µL channels. This follows the DPX Hamilton procedure from **step 4**. Steps 1–3 (pick up the piercing tool, pierce the barrier, park the tool) are not in this method. Pierce every SEC tip by hand before the run and check that the opening is large enough for a 300 µL tip to pass through.

The column example in the DPX procedure is a **250 µL, 5,000 Da** tip, which uses **688 µL** of swelling buffer (the 250–300 µL void row in their table). That is the default. Change the column preset if you have a different tip.

Watch the first swell with a hand near stop. Two heights are set on the real deck rather than taken from a liquid class:

- **Buffer / sample / flow height** is millimetres above the well bottom. The tip stays at that fixed height (no liquid-level detection).
- **Load depth** is how far a 300 µL tip goes below the SEC tip opening, just under the pierced barrier. Use **Teach load height** on an empty tip before the first sample.

Hamilton’s pipetting settle field stops at 9.9 s. The 150 s swell and 30 s column settles are timed in Python while the tip is left at that height. The hanging-drop wait is the same kind of pause, done in the air after the SEC tip is picked up.

Labware is a `TIP_CAR_480_A00` tip carrier and a `PLT_CAR_L5AC_A00` plate carrier (Hamilton 182090, the deep-well carrier in the DPX procedure). Swelling buffer is in a Corning 96-well 2 mL deep-well plate (`Cor_96_wellplate_2mL_Vb`), one well per sample, instead of the 300 mL reservoir. Sample, waste, and elution use that same plate. Slot 1 is the front of the carrier and slot 5 is the back. Set the rail and the slot for each rack and plate before Setup Deck.

Once this works on its own, the SEC sample plate can be fed from the nickel elution plate in `phytip_hamilton_GUI.ipynb`.


In [ ]:
import asyncio
import time

import ipywidgets as widgets
from IPython.display import display
import sys
sys.path.insert(0, r"C:\Users\16122\Documents\Github\pylabrobot")
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.liquid_handling.liquid_classes.hamilton import HamiltonLiquidClass
from pylabrobot.resources import (
    Coordinate,
    Cor_96_wellplate_2mL_Vb,
    PLT_CAR_L5AC_A00,
    Resource,
    TIP_CAR_480_A00,
    Well,
    hamilton_96_tiprack_300uL,
    hamilton_96_tiprack_1000uL_filter_ultrawide,
    set_volume_tracking,
)
from pylabrobot.resources.hamilton import STARDeck

deck_is_ready = False
lh = None
LOAD_WELLS = {}
teach_tip_spot = None
status_output = None


In [ ]:
# DPX Table 1. "Void" is the top of each bin, which matches the catalog sample volume.
# The procedure's worked example (250 uL column, 5,000 Da, 688 uL swell) is the last row.
SEC_COLUMNS = {
    "1500 Da, void 100-150 uL (swell 263)": {"void_uL": 150, "swell_uL": 263},
    "1500 Da, void 150-200 uL (swell 394)": {"void_uL": 200, "swell_uL": 394},
    "1500 Da, void 200-250 uL (swell 525)": {"void_uL": 250, "swell_uL": 525},
    "1500 Da, void 250-300 uL (swell 656)": {"void_uL": 300, "swell_uL": 656},
    "5000 Da, void 100-150 uL (swell 275)": {"void_uL": 150, "swell_uL": 275},
    "5000 Da, void 150-200 uL (swell 408)": {"void_uL": 200, "swell_uL": 408},
    "5000 Da, void 200-250 uL (swell 550)": {"void_uL": 250, "swell_uL": 550},
    "5000 Da, void 250-300 uL (swell 688)": {"void_uL": 300, "swell_uL": 688},
}
DEFAULT_COLUMN = "5000 Da, void 250-300 uL (swell 688)"

number_of_samples = 8
first_well = "A1"
swell_uL = 688
void_uL = 300
sample_uL = 250
elution_uL = 250
correction_uL = 0
do_void_elution = True
do_buffer_touchoff = True

column_flow = 5.0  # uL/s through the SEC tip
transfer_flow = 75.0  # uL/s when dispensing into the SEC tip (DPX: 50-100)
aspirate_flow = 100.0  # uL/s for sample and buffer pickup with 300 uL tips

swell_hold_s = 150.0
flow_hold_s = 30.0
drop_hold_s = 10.0
touch_hold_s = 5.0
drop_uL = 25.0
touch_air_uL = 20.0
airgap_margin_uL = 50.0

# Positive load depth puts the 300 uL tip that many mm below the SEC opening.
load_depth_mm = 6.0
buffer_height_mm = 5.0
sample_height_mm = 2.0
flow_height_mm = 15.0

tip_carrier_rail = 7
plate_carrier_rail = 1
# Carrier slots are 1 at the front through 5 at the back.
sec_tip_pos = 1
sample_tip_pos = 2
buffer_tip_pos = 3
buffer_plate_pos = 1
sample_plate_pos = 2
waste_plate_pos = 3
elution_plate_pos = 4
DEVELOPMENT_MODE = False

FIRMWARE_MAX_SETTLE_S = 9.9


In [ ]:
def log(message: str) -> None:
    if status_output is not None:
        status_output.append_stdout(message + "\n")
    else:
        print(message)


def plate_wells(plate_name: str, names: list[str]):
    plate = lh.deck.get_resource(plate_name)
    return [plate[name][0] for name in names]


def tip_spots(rack_name: str, names: list[str]):
    rack = lh.deck.get_resource(rack_name)
    return [rack[name][0] for name in names]


def sample_names(n: int, start: str) -> list[str]:
    rows = list("ABCDEFGH")
    start = start.strip().upper()
    row_i = rows.index(start[0])
    col = int(start[1:])
    names = []
    for column in range(col, 13):
        first_row = row_i if column == col else 0
        for row in rows[first_row:]:
            names.append(f"{row}{column}")
            if len(names) == n:
                return names
    raise ValueError(f"Only {len(names)} wells are left from {start}.")


def column_groups(names: list[str]) -> list[list[str]]:
    """Split well names into same-column groups so the 8 channels share one X."""
    groups: list[list[str]] = []
    current: list[str] = []
    for name in names:
        if current and name[1:] != current[0][1:]:
            groups.append(current)
            current = []
        current.append(name)
    if current:
        groups.append(current)
    return groups


def repeat(value, n: int):
    return [value] * n


def end_z(resource, liquid_height: float = 0.0, z_offset: float = 0.0) -> float:
    """Absolute Z of the tip end, using the same origin as STAR aspirate/dispense."""
    return (
        resource.get_location_wrt(lh.deck).z
        + resource.material_z_thickness
        + z_offset
        + liquid_height
    )


def zero_air_liquid_class(dispense_transport_air: float) -> HamiltonLiquidClass:
    """Identity correction and no extra air, so the programmed microlitres are the plunger move."""
    return HamiltonLiquidClass(
        curve={0.0: 0.0, 1000.0: 1000.0},
        aspiration_flow_rate=column_flow,
        aspiration_mix_flow_rate=column_flow,
        aspiration_air_transport_volume=0.0,
        aspiration_blow_out_volume=0.0,
        aspiration_swap_speed=10.0,
        aspiration_settling_time=0.0,
        aspiration_over_aspirate_volume=0.0,
        aspiration_clot_retract_height=0.0,
        dispense_flow_rate=column_flow,
        dispense_mode=2.0,
        dispense_mix_flow_rate=column_flow,
        dispense_air_transport_volume=dispense_transport_air,
        dispense_blow_out_volume=0.0,
        dispense_swap_speed=10.0,
        dispense_settling_time=0.0,
        dispense_stop_flow_rate=5.0,
        dispense_stop_back_volume=0.0,
    )


def build_load_guide(sec_rack):
    """Wells whose bottoms sit at the SEC tip opening, aligned with each tip spot.

    STAR pickup puts the tip bottom at the tip-spot origin and the opening
    ``tip_length - fitting_depth`` above that. Dispense into these wells with a
    negative Z offset to land just below the pierced barrier.
    """
    prototype = sec_rack["A1"][0].get_tip()
    opening_above_spot = prototype.total_tip_length - prototype.fitting_depth
    guide = Resource(
        name="sec_load_guide",
        size_x=122.4,
        size_y=82.6,
        size_z=20.0,
        category="plate",
    )
    wells = {}
    for row in "ABCDEFGH":
        for col in range(1, 13):
            spot = sec_rack[f"{row}{col}"][0]
            well = Well(
                name=f"sec_load_{row}{col}",
                size_x=9.0,
                size_y=9.0,
                size_z=1.0,
                material_z_thickness=0.0,
                max_volume=1000,
            )
            guide.assign_child_resource(
                well,
                location=Coordinate(
                    spot.location.x,
                    spot.location.y,
                    spot.location.z + opening_above_spot,
                ),
            )
            wells[f"{row}{col}"] = well
    return guide, wells, prototype, opening_above_spot


def place_on_carrier(carrier, placements, kind: str) -> None:
    """Place resources on carrier slots numbered 1 (front) through 5 (back)."""
    positions = [position for position, _resource in placements]
    if any(position < 1 or position > 5 for position in positions):
        raise ValueError(f"{kind} positions must be 1 (front) through 5 (back).")
    if len(set(positions)) != len(positions):
        raise ValueError(f"{kind} positions overlap: {positions}.")
    for position, resource in placements:
        carrier[position - 1] = resource


async def setup_deck():
    global lh, deck_is_ready, LOAD_WELLS

    if deck_is_ready or lh is not None:
        return "Deck setup already started. Restart the kernel to set it up again."

    lh = LiquidHandler(
        backend=STARBackend(read_timeout=2400),
        deck=STARDeck(core_grippers="1000uL-5mL-on-waste"),
    )
    await lh.setup(skip_iswap=True)
    # Tip presence has to stay on so parked SEC tips can be picked up again.
    # Volume tracking stays off: the SEC dispense pushes air, not a tracked liquid.
    set_volume_tracking(enabled=False)

    lh.deck.get_resource("trash_core96").location = Coordinate(-260, 106, 216.4)

    tip_car = TIP_CAR_480_A00(name="tip carrier")
    place_on_carrier(
        tip_car,
        [
            (sec_tip_pos, hamilton_96_tiprack_1000uL_filter_ultrawide(name="sec_tips")),
            (sample_tip_pos, hamilton_96_tiprack_300uL(name="sample_transfer_tips")),
            (buffer_tip_pos, hamilton_96_tiprack_300uL(name="buffer_transfer_tips")),
        ],
        "Tip",
    )
    lh.deck.assign_child_resource(tip_car, rails=tip_carrier_rail)

    plt_car = PLT_CAR_L5AC_A00(name="sec_plate_carrier")
    place_on_carrier(
        plt_car,
        [
            (buffer_plate_pos, Cor_96_wellplate_2mL_Vb(name="sec_buffer_plate")),
            (sample_plate_pos, Cor_96_wellplate_2mL_Vb(name="sec_sample_plate")),
            (waste_plate_pos, Cor_96_wellplate_2mL_Vb(name="sec_waste_plate")),
            (elution_plate_pos, Cor_96_wellplate_2mL_Vb(name="sec_elution_plate")),
        ],
        "Plate",
    )
    lh.deck.assign_child_resource(plt_car, rails=plate_carrier_rail)

    sec_rack = lh.deck.get_resource("sec_tips")
    guide, LOAD_WELLS, prototype, opening_above_spot = build_load_guide(sec_rack)
    # Parent the guide to the tip rack. A deck-level copy sits inside the carrier
    # footprint and STARDeck rejects that as a collision.
    sec_rack.assign_child_resource(guide, location=Coordinate.zero())

    deck_is_ready = True
    summary = lh.summary()
    note = (
        f"SEC tip opening is {opening_above_spot:.2f} mm above the tip-spot origin "
        f"(length {prototype.total_tip_length:.2f} mm, fitting depth {prototype.fitting_depth:.2f} mm). "
        "Load depth is measured down from that opening."
    )
    if isinstance(summary, str):
        return "Deck setup complete.\n\n" + summary + "\n\n" + note
    return "Deck setup complete.\n\n" + note


In [ ]:
async def hold(seconds: float, message: str) -> None:
    log(message)
    remaining = float(seconds)
    while remaining > 0:
        step = min(30.0, remaining)
        await asyncio.sleep(step)
        remaining -= step
        if remaining > 0:
            log(f"  {remaining:.0f} s remaining")


async def set_plunger(n_channels: int, volume: float, flow_rate: float) -> None:
    for channel in range(n_channels):
        await lh.backend.channel_dispensing_drive_move_to_volume_position(
            channel_idx=channel,
            vol=volume,
            flow_rate=flow_rate,
        )


async def plunger_positions(n_channels: int) -> list[float]:
    return [
        await lh.backend.channel_dispensing_drive_request_position(channel_idx=channel)
        for channel in range(n_channels)
    ]


async def aspirate_hanging_drop(n_channels: int, airgap: float) -> None:
    """Pull the hanging drop in. Restore the air gap first if pickup homed the drive."""
    for channel in range(n_channels):
        current = await lh.backend.channel_dispensing_drive_request_position(channel_idx=channel)
        if current < airgap - 15:
            # C0TP homes the dosing drive. There is no pick_up_tips argument that
            # disables it, so the air has to be drawn again with the tip mounted.
            log(
                f"Channel {channel} plunger is at {current:.0f} uL after tip pickup "
                f"(air gap was {airgap:.0f} uL). Restoring it at {column_flow:.1f} uL/s "
                "with the SEC tip on. That air comes in through the frit."
            )
            await lh.backend.channel_dispensing_drive_move_to_volume_position(
                channel_idx=channel,
                vol=airgap,
                flow_rate=column_flow,
            )
            current = airgap
        await lh.backend.channel_dispensing_drive_move_to_volume_position(
            channel_idx=channel,
            vol=current + drop_uL,
            flow_rate=column_flow,
        )
    await hold(drop_hold_s, f"Waiting {drop_hold_s:.0f} s for the hanging drop to enter the frit.")


def airgap_volume(push_uL: float) -> float:
    # Greater than the volume that will be pushed through the column, plus the drop
    # aspiration and the touch-off air that happen while the tip is on.
    return push_uL + drop_uL + touch_air_uL + airgap_margin_uL


async def swell_columns(sec_spots, buffer_wells) -> None:
    n = len(buffer_wells)
    liquid = zero_air_liquid_class(0.0)
    z = end_z(buffer_wells[0], buffer_height_mm)
    log(f"Step 4-5. Pick up SEC tips and aspirate {swell_uL:.0f} uL swelling buffer at {column_flow:.1f} uL/s.")
    await lh.pick_up_tips(sec_spots)
    await lh.aspirate(
        buffer_wells,
        vols=repeat(float(swell_uL), n),
        flow_rates=repeat(column_flow, n),
        liquid_height=repeat(buffer_height_mm, n),
        hamilton_liquid_classes=repeat(liquid, n),
        disable_volume_correction=repeat(True, n),
        transport_air_volume=repeat(0.0, n),
        pull_out_distance_transport_air=repeat(0.0, n),
        swap_speed=repeat(10.0, n),
        settling_time=repeat(0.0, n),
        min_z_endpos=z,
    )
    await hold(
        swell_hold_s,
        f"Step 6. Holding SEC tips in the swelling buffer for {swell_hold_s:.0f} s. "
        "The tips should still be submerged. Stop the run if they have retracted.",
    )
    if do_buffer_touchoff:
        log("Step 7. Touch off in the buffer so a hanging drop wicks off the SEC tip.")
        await lh.dispense(
            buffer_wells,
            vols=repeat(0.0, n),
            flow_rates=repeat(column_flow, n),
            liquid_height=repeat(buffer_height_mm, n),
            hamilton_liquid_classes=repeat(liquid, n),
            disable_volume_correction=repeat(True, n),
            transport_air_volume=repeat(0.0, n),
            pull_out_distance_transport_air=repeat(0.0, n),
            swap_speed=repeat(10.0, n),
            settling_time=repeat(0.0, n),
            minimum_traverse_height_at_beginning_of_a_command=z,
        )
    await lh.drop_tips(sec_spots, allow_nonzero_volume=True)


async def dispense_into_sec_tips(transfer_spots, source_wells, load_wells, volume: float, what: str) -> None:
    n = len(load_wells)
    liquid = zero_air_liquid_class(0.0)
    log(f"Pick up 300 uL tips and aspirate {volume:.0f} uL {what}.")
    await lh.pick_up_tips(transfer_spots)
    await lh.aspirate(
        source_wells,
        vols=repeat(float(volume), n),
        flow_rates=repeat(aspirate_flow, n),
        liquid_height=repeat(sample_height_mm if what == "sample" else buffer_height_mm, n),
        hamilton_liquid_classes=repeat(liquid, n),
        disable_volume_correction=repeat(True, n),
        transport_air_volume=repeat(0.0, n),
        pull_out_distance_transport_air=repeat(0.0, n),
        swap_speed=repeat(10.0, n),
    )
    log(
        f"Dispense {what} into the SEC tips at {transfer_flow:.0f} uL/s, "
        f"{load_depth_mm:.1f} mm below the opening."
    )
    await lh.dispense(
        load_wells,
        vols=repeat(float(volume), n),
        flow_rates=repeat(transfer_flow, n),
        offsets=repeat(Coordinate(0, 0, -load_depth_mm), n),
        hamilton_liquid_classes=repeat(liquid, n),
        disable_volume_correction=repeat(True, n),
        transport_air_volume=repeat(0.0, n),
        pull_out_distance_transport_air=repeat(0.0, n),
        swap_speed=repeat(10.0, n),
    )
    if what == "sample":
        await lh.discard_tips()
    else:
        await lh.drop_tips(transfer_spots, allow_nonzero_volume=True)


async def push_through(sec_spots, dest_wells, liquid_uL: float, where: str) -> None:
    """Pre-load an air gap, pick up the SEC tips, pull the drop in, then push `liquid_uL + correction`."""
    n = len(dest_wells)
    push_uL = float(liquid_uL) + float(correction_uL)
    airgap = airgap_volume(push_uL)
    if airgap + drop_uL > 1200:
        raise ValueError(f"Air gap {airgap:.0f} uL plus the drop does not fit in a 1000 uL channel.")
    flow_liquid = zero_air_liquid_class(0.0)
    touch_liquid = zero_air_liquid_class(touch_air_uL)
    z = end_z(dest_wells[0], flow_height_mm)

    log(f"Move plungers to a {airgap:.0f} uL air gap before picking up the SEC tips.")
    await set_plunger(n, airgap, flow_rate=100.0)
    before = await plunger_positions(n)
    log("Plunger before pickup: " + ", ".join(f"ch{i} {p:.0f} uL" for i, p in enumerate(before)))
    # pick_up_tips backend_kwargs that reach the STAR are only the Z window,
    # traverse height, and pickup_method (out of rack / out of wash). None of
    # them keep the dosing drive. An unknown kwarg is dropped before C0TP.
    await lh.pick_up_tips(sec_spots)
    after = await plunger_positions(n)
    log("Plunger after pickup: " + ", ".join(f"ch{i} {p:.0f} uL" for i, p in enumerate(after)))
    log(f"Aspirate {drop_uL:.0f} uL at {column_flow:.1f} uL/s to pick up the hanging drop.")
    await aspirate_hanging_drop(n, airgap)

    log(
        f"Push {push_uL:.0f} uL through the column into the {where} "
        f"({liquid_uL:.0f} uL liquid + {correction_uL:.0f} uL correction) at {column_flow:.1f} uL/s."
    )
    await lh.dispense(
        dest_wells,
        vols=repeat(push_uL, n),
        flow_rates=repeat(column_flow, n),
        liquid_height=repeat(flow_height_mm, n),
        hamilton_liquid_classes=repeat(flow_liquid, n),
        disable_volume_correction=repeat(True, n),
        transport_air_volume=repeat(0.0, n),
        pull_out_distance_transport_air=repeat(0.0, n),
        swap_speed=repeat(10.0, n),
        settling_time=repeat(0.0, n),
        min_z_endpos=z,
    )
    await hold(flow_hold_s, f"Settling {flow_hold_s:.0f} s after the column flow. Tips should still be in the {where}.")
    log(f"Touch off into the {where} with {touch_air_uL:.0f} uL transport air, then wait {touch_hold_s:.0f} s.")
    await lh.dispense(
        dest_wells,
        vols=repeat(0.0, n),
        flow_rates=repeat(column_flow, n),
        liquid_height=repeat(flow_height_mm, n),
        hamilton_liquid_classes=repeat(touch_liquid, n),
        disable_volume_correction=repeat(True, n),
        transport_air_volume=repeat(touch_air_uL, n),
        pull_out_distance_transport_air=repeat(2.0, n),
        swap_speed=repeat(10.0, n),
        settling_time=repeat(min(touch_hold_s, FIRMWARE_MAX_SETTLE_S), n),
        minimum_traverse_height_at_beginning_of_a_command=z,
    )
    extra_wait = touch_hold_s - FIRMWARE_MAX_SETTLE_S
    if extra_wait > 0:
        await hold(extra_wait, f"Waiting the rest of the {touch_hold_s:.0f} s touch-off.")
    await lh.drop_tips(sec_spots, allow_nonzero_volume=True)


async def process_group(names: list[str], sample_tip_col: int, buffer_tip_col: int) -> tuple[int, int]:
    rows = [name[0] for name in names]
    sec_spots = tip_spots("sec_tips", names)
    buffer_wells = plate_wells("sec_buffer_plate", names)
    sample_wells = plate_wells("sec_sample_plate", names)
    waste_wells = plate_wells("sec_waste_plate", names)
    elution_wells = plate_wells("sec_elution_plate", names)
    load_wells = [LOAD_WELLS[name] for name in names]
    log(f"Column {names[0][1:]} wells {names[0]}:{names[-1]}")

    await swell_columns(sec_spots, buffer_wells)

    sample_names_for_tips = [f"{row}{sample_tip_col}" for row in rows]
    if sample_tip_col > 12:
        raise RuntimeError("Not enough 300 uL sample-transfer tips.")
    log("Step 8-11. Transfer sample into the pierced SEC tips.")
    await dispense_into_sec_tips(
        tip_spots("sample_transfer_tips", sample_names_for_tips),
        sample_wells,
        load_wells,
        sample_uL,
        "sample",
    )
    sample_tip_col += 1

    log("Step 12-17. Flow the sample into the column. Void buffer collects in the waste plate.")
    await push_through(sec_spots, waste_wells, sample_uL, "waste plate")

    void_liquid = max(0.0, float(void_uL) - float(sample_uL)) if do_void_elution else 0.0
    if do_void_elution and void_liquid <= 0:
        log("Sample volume is at least the void volume, so the void elution is skipped.")
    elif void_liquid > 0:
        if buffer_tip_col > 12:
            raise RuntimeError("Not enough 300 uL buffer-transfer tips.")
        buffer_tip_names = [f"{row}{buffer_tip_col}" for row in rows]
        log(
            f"Void elution. Sample {sample_uL:.0f} uL is below the {void_uL:.0f} uL void, "
            f"so {void_liquid:.0f} uL of buffer is pushed to waste before collection."
        )
        await dispense_into_sec_tips(
            tip_spots("buffer_transfer_tips", buffer_tip_names),
            buffer_wells,
            load_wells,
            void_liquid,
            "void buffer",
        )
        await push_through(sec_spots, waste_wells, void_liquid, "waste plate")

    if buffer_tip_col > 12:
        raise RuntimeError("Not enough 300 uL buffer-transfer tips.")
    buffer_tip_names = [f"{row}{buffer_tip_col}" for row in rows]
    log("Step 18-26. Add elution buffer and collect the target in the elution plate.")
    await dispense_into_sec_tips(
        tip_spots("buffer_transfer_tips", buffer_tip_names),
        buffer_wells,
        load_wells,
        elution_uL,
        "elution buffer",
    )
    buffer_tip_col += 1
    await push_through(sec_spots, elution_wells, elution_uL, "elution plate")
    return sample_tip_col, buffer_tip_col


async def run_sec() -> None:
    if not deck_is_ready or lh is None:
        raise RuntimeError("Deck is not ready. Click Setup Deck first.")
    if any(tracker.has_tip for tracker in lh.head.values()):
        raise RuntimeError("A channel still has a tip. Return or trash it before running.")

    names = sample_names(number_of_samples, first_well)
    void_liquid = max(0.0, float(void_uL) - float(sample_uL)) if do_void_elution else 0.0
    buffer_need = swell_uL + void_liquid + elution_uL + 200
    log("Piercing is skipped. SEC barriers must already be open.")
    log(
        f"Fill each used buffer well with at least {buffer_need:.0f} uL "
        f"({swell_uL:.0f} swell + {void_liquid:.0f} void elution + {elution_uL:.0f} elution + 200 uL extra)."
    )
    log(f"Samples: {', '.join(names)}")
    sample_tip_col = 1
    buffer_tip_col = 1
    for group in column_groups(names):
        sample_tip_col, buffer_tip_col = await process_group(group, sample_tip_col, buffer_tip_col)
    log("Step 27. SEC method finished. SEC tips are back in the rack.")


async def teach_load_height() -> None:
    """Park a 300 uL tip just below the SEC opening and leave it there so the height can be checked."""
    global teach_tip_spot
    if not deck_is_ready or lh is None:
        raise RuntimeError("Deck is not ready. Click Setup Deck first.")
    if any(tracker.has_tip for tracker in lh.head.values()):
        raise RuntimeError("A channel still has a tip. Return or trash it first.")
    names = sample_names(1, first_well)
    teach_tip_spot = tip_spots("sample_transfer_tips", names)[0]
    load_well = LOAD_WELLS[names[0]]
    z = end_z(load_well, z_offset=-load_depth_mm)
    liquid = zero_air_liquid_class(0.0)
    log(f"Teaching load height at {names[0]}, {load_depth_mm:.1f} mm below the SEC opening (tip end Z {z:.2f} mm).")
    await lh.pick_up_tips([teach_tip_spot])
    await lh.dispense(
        [load_well],
        vols=[0.0],
        flow_rates=[transfer_flow],
        offsets=[Coordinate(0, 0, -load_depth_mm)],
        hamilton_liquid_classes=[liquid],
        disable_volume_correction=[True],
        transport_air_volume=[0.0],
        pull_out_distance_transport_air=[0.0],
        min_z_endpos=z,
    )
    log("Tip is holding at the load height. Look at the pierced barrier, then click Return teach tip.")


async def return_teach_tip() -> None:
    global teach_tip_spot
    if teach_tip_spot is None:
        raise RuntimeError("No teach tip is tracked. Use Return 8-head tips if a tip is still on a channel.")
    await lh.drop_tips([teach_tip_spot], allow_nonzero_volume=True)
    teach_tip_spot = None
    log("Returned the teach tip.")


In [ ]:
def apply_ui_settings(settings: dict) -> None:
    global number_of_samples, first_well
    global swell_uL, void_uL, sample_uL, elution_uL, correction_uL
    global do_void_elution, do_buffer_touchoff
    global column_flow, transfer_flow, aspirate_flow
    global swell_hold_s, flow_hold_s, drop_hold_s, touch_hold_s
    global drop_uL, touch_air_uL, airgap_margin_uL
    global load_depth_mm, buffer_height_mm, sample_height_mm, flow_height_mm
    global tip_carrier_rail, plate_carrier_rail, DEVELOPMENT_MODE
    global sec_tip_pos, sample_tip_pos, buffer_tip_pos
    global buffer_plate_pos, sample_plate_pos, waste_plate_pos, elution_plate_pos

    samples = int(settings["number_of_samples"])
    if samples < 1 or samples > 96:
        raise ValueError("Samples must be between 1 and 96.")
    first = str(settings["first_well"]).strip().upper()
    if len(first) < 2 or first[0] not in "ABCDEFGH" or not first[1:].isdigit():
        raise ValueError("First well must look like A1 to H12.")
    first_col = int(first[1:])
    if first_col < 1 or first_col > 12:
        raise ValueError("First well must be in columns 1 to 12.")
    rows_left = 8 - "ABCDEFGH".index(first[0])
    wells_left = rows_left + (12 - first_col) * 8
    if samples > wells_left:
        raise ValueError(f"With first well {first}, the maximum number of samples is {wells_left}.")

    sample = float(settings["sample_uL"])
    elution = float(settings["elution_uL"])
    if sample > 300 or elution > 350:
        raise ValueError("Sample must be <= 300 uL and elution <= 350 uL for the 300 uL transfer tips.")
    if float(settings["touch_air_uL"]) > 50:
        raise ValueError("Touch-off transport air must be <= 50 uL (Hamilton limit).")
    if float(settings["drop_uL"]) < 20 or float(settings["drop_uL"]) > 30:
        raise ValueError("Hanging-drop aspiration should stay in the DPX range of 20-30 uL.")

    number_of_samples = samples
    first_well = first
    swell_uL = float(settings["swell_uL"])
    void_uL = float(settings["void_uL"])
    sample_uL = sample
    elution_uL = elution
    correction_uL = float(settings["correction_uL"])
    do_void_elution = bool(settings["do_void_elution"])
    do_buffer_touchoff = bool(settings["do_buffer_touchoff"])
    column_flow = float(settings["column_flow"])
    transfer_flow = float(settings["transfer_flow"])
    aspirate_flow = float(settings["aspirate_flow"])
    swell_hold_s = float(settings["swell_hold_s"])
    flow_hold_s = float(settings["flow_hold_s"])
    drop_hold_s = float(settings["drop_hold_s"])
    touch_hold_s = float(settings["touch_hold_s"])
    drop_uL = float(settings["drop_uL"])
    touch_air_uL = float(settings["touch_air_uL"])
    airgap_margin_uL = float(settings["airgap_margin_uL"])
    load_depth_mm = float(settings["load_depth_mm"])
    buffer_height_mm = float(settings["buffer_height_mm"])
    sample_height_mm = float(settings["sample_height_mm"])
    flow_height_mm = float(settings["flow_height_mm"])
    tip_positions = [
        int(settings["sec_tip_pos"]),
        int(settings["sample_tip_pos"]),
        int(settings["buffer_tip_pos"]),
    ]
    plate_positions = [
        int(settings["buffer_plate_pos"]),
        int(settings["sample_plate_pos"]),
        int(settings["waste_plate_pos"]),
        int(settings["elution_plate_pos"]),
    ]
    for label, positions in (("Tip", tip_positions), ("Plate", plate_positions)):
        if any(position < 1 or position > 5 for position in positions):
            raise ValueError(f"{label} positions must be 1 (front) through 5 (back).")
        if len(set(positions)) != len(positions):
            raise ValueError(f"{label} positions overlap: {positions}.")

    tip_carrier_rail = int(settings["tip_carrier_rail"])
    plate_carrier_rail = int(settings["plate_carrier_rail"])
    sec_tip_pos, sample_tip_pos, buffer_tip_pos = tip_positions
    buffer_plate_pos, sample_plate_pos, waste_plate_pos, elution_plate_pos = plate_positions
    DEVELOPMENT_MODE = bool(settings["development_mode"])


title = widgets.HTML("<h3>DPX SEC on Hamilton</h3>")
instructions = widgets.HTML(
    "<b>Before the run:</b> pierce the SEC barriers by hand. "
    "Plate carrier is Hamilton 182090 (<code>PLT_CAR_L5AC_A00</code>). "
    "Swelling buffer is a Corning 96-well 2 mL deep-well plate, one well per sample. "
    "Slot 1 is the front of the carrier and slot 5 is the back.<br>"
    "<b>Workflow:</b> 1) Set the column, volumes, and carrier slots, 2) Setup Deck, "
    "3) Teach load height and look at the tip, 4) Return that tip, 5) Run SEC."
)

column_field = widgets.Dropdown(
    options=list(SEC_COLUMNS.keys()),
    value=DEFAULT_COLUMN,
    description="Column",
    style={"description_width": "140px"},
    layout={"width": "520px"},
)
samples_field = widgets.BoundedIntText(value=number_of_samples, min=1, max=96, description="Samples", style={"description_width": "140px"})
first_well_field = widgets.Text(value=first_well, description="First well", style={"description_width": "140px"})
swell_field = widgets.BoundedFloatText(value=swell_uL, min=50, max=900, step=1, description="Swell uL", style={"description_width": "140px"})
void_field = widgets.BoundedFloatText(value=void_uL, min=50, max=400, step=1, description="Void uL", style={"description_width": "140px"})
sample_field = widgets.BoundedFloatText(value=sample_uL, min=1, max=300, step=1, description="Sample uL", style={"description_width": "140px"})
elution_field = widgets.BoundedFloatText(value=elution_uL, min=1, max=350, step=1, description="Elution uL", style={"description_width": "140px"})
correction_field = widgets.BoundedFloatText(value=correction_uL, min=0, max=200, step=1, description="Correction uL", style={"description_width": "140px"})
void_toggle = widgets.Checkbox(value=do_void_elution, description="Void elution when sample < void")
touch_toggle = widgets.Checkbox(value=do_buffer_touchoff, description="Touch off in buffer after swell")
development_toggle = widgets.Checkbox(value=False, description="Development mode (verbose logs)")

column_flow_field = widgets.BoundedFloatText(value=column_flow, min=1, max=50, step=0.5, description="Column uL/s", style={"description_width": "140px"})
transfer_flow_field = widgets.BoundedFloatText(value=transfer_flow, min=20, max=250, step=5, description="Into-tip uL/s", style={"description_width": "140px"})
aspirate_flow_field = widgets.BoundedFloatText(value=aspirate_flow, min=10, max=400, step=5, description="Aspirate uL/s", style={"description_width": "140px"})
swell_hold_field = widgets.BoundedFloatText(value=swell_hold_s, min=0, max=600, step=1, description="Swell hold s", style={"description_width": "140px"})
flow_hold_field = widgets.BoundedFloatText(value=flow_hold_s, min=0, max=120, step=1, description="Flow hold s", style={"description_width": "140px"})
drop_hold_field = widgets.BoundedFloatText(value=drop_hold_s, min=0, max=30, step=0.5, description="Drop hold s", style={"description_width": "140px"})
touch_hold_field = widgets.BoundedFloatText(value=touch_hold_s, min=0, max=30, step=0.5, description="Touch hold s", style={"description_width": "140px"})
drop_field = widgets.BoundedFloatText(value=drop_uL, min=20, max=30, step=1, description="Drop uL", style={"description_width": "140px"})
touch_air_field = widgets.BoundedFloatText(value=touch_air_uL, min=0, max=50, step=1, description="Touch air uL", style={"description_width": "140px"})
margin_field = widgets.BoundedFloatText(value=airgap_margin_uL, min=0, max=200, step=5, description="Air-gap extra", style={"description_width": "140px"})
load_depth_field = widgets.BoundedFloatText(value=load_depth_mm, min=-10, max=40, step=0.5, description="Load depth mm", style={"description_width": "140px"})
buffer_height_field = widgets.BoundedFloatText(value=buffer_height_mm, min=1, max=30, step=0.5, description="Buffer height", style={"description_width": "140px"})
sample_height_field = widgets.BoundedFloatText(value=sample_height_mm, min=0.5, max=20, step=0.5, description="Sample height", style={"description_width": "140px"})
flow_height_field = widgets.BoundedFloatText(value=flow_height_mm, min=1, max=40, step=0.5, description="Flow height", style={"description_width": "140px"})
tip_rail_field = widgets.BoundedIntText(value=tip_carrier_rail, min=-20, max=60, description="Tip rail", style={"description_width": "140px"})
plate_rail_field = widgets.BoundedIntText(value=plate_carrier_rail, min=-20, max=60, description="Plate rail", style={"description_width": "140px"})
sec_tip_pos_field = widgets.BoundedIntText(value=sec_tip_pos, min=1, max=5, description="SEC tips", style={"description_width": "140px"})
sample_tip_pos_field = widgets.BoundedIntText(value=sample_tip_pos, min=1, max=5, description="Sample tips", style={"description_width": "140px"})
buffer_tip_pos_field = widgets.BoundedIntText(value=buffer_tip_pos, min=1, max=5, description="Buffer tips", style={"description_width": "140px"})
buffer_plate_pos_field = widgets.BoundedIntText(value=buffer_plate_pos, min=1, max=5, description="Swell plate", style={"description_width": "140px"})
sample_plate_pos_field = widgets.BoundedIntText(value=sample_plate_pos, min=1, max=5, description="Sample plate", style={"description_width": "140px"})
waste_plate_pos_field = widgets.BoundedIntText(value=waste_plate_pos, min=1, max=5, description="Waste plate", style={"description_width": "140px"})
elution_plate_pos_field = widgets.BoundedIntText(value=elution_plate_pos, min=1, max=5, description="Elution plate", style={"description_width": "140px"})


def _on_column_changed(change):
    preset = SEC_COLUMNS[change["new"]]
    swell_field.value = preset["swell_uL"]
    void_field.value = preset["void_uL"]


column_field.observe(_on_column_changed, names="value")

setup_button = widgets.Button(description="Setup Deck", button_style="info")
teach_button = widgets.Button(description="Teach load height", button_style="warning", disabled=True)
return_teach_button = widgets.Button(description="Return teach tip", button_style="warning", disabled=True)
run_button = widgets.Button(description="Run SEC", button_style="success", disabled=True)
return_8_button = widgets.Button(description="Return 8-head tips", button_style="warning", disabled=True)
trash_8_button = widgets.Button(description="Trash 8-head tips", button_style="danger", disabled=True)

status_output = widgets.Output(layout={"border": "1px solid #ddd", "padding": "8px", "max_height": "360px", "overflow": "auto"})
deck_summary = widgets.Textarea(
    value="",
    description="Deck",
    layout={"width": "100%", "height": "220px"},
    disabled=True,
)
run_state_label = widgets.HTML("")


def _collect_settings():
    return {
        "number_of_samples": samples_field.value,
        "first_well": first_well_field.value,
        "swell_uL": swell_field.value,
        "void_uL": void_field.value,
        "sample_uL": sample_field.value,
        "elution_uL": elution_field.value,
        "correction_uL": correction_field.value,
        "do_void_elution": void_toggle.value,
        "do_buffer_touchoff": touch_toggle.value,
        "column_flow": column_flow_field.value,
        "transfer_flow": transfer_flow_field.value,
        "aspirate_flow": aspirate_flow_field.value,
        "swell_hold_s": swell_hold_field.value,
        "flow_hold_s": flow_hold_field.value,
        "drop_hold_s": drop_hold_field.value,
        "touch_hold_s": touch_hold_field.value,
        "drop_uL": drop_field.value,
        "touch_air_uL": touch_air_field.value,
        "airgap_margin_uL": margin_field.value,
        "load_depth_mm": load_depth_field.value,
        "buffer_height_mm": buffer_height_field.value,
        "sample_height_mm": sample_height_field.value,
        "flow_height_mm": flow_height_field.value,
        "tip_carrier_rail": tip_rail_field.value,
        "plate_carrier_rail": plate_rail_field.value,
        "sec_tip_pos": sec_tip_pos_field.value,
        "sample_tip_pos": sample_tip_pos_field.value,
        "buffer_tip_pos": buffer_tip_pos_field.value,
        "buffer_plate_pos": buffer_plate_pos_field.value,
        "sample_plate_pos": sample_plate_pos_field.value,
        "waste_plate_pos": waste_plate_pos_field.value,
        "elution_plate_pos": elution_plate_pos_field.value,
        "development_mode": development_toggle.value,
    }


def _set_busy(busy: bool) -> None:
    setup_button.disabled = busy
    teach_button.disabled = busy or not deck_is_ready
    return_teach_button.disabled = busy or not deck_is_ready
    run_button.disabled = busy or not deck_is_ready
    return_8_button.disabled = busy or not deck_is_ready
    trash_8_button.disabled = busy or not deck_is_ready


def _show_error(exc: Exception) -> None:
    status_output.append_stdout(f"Failed: {exc}\n")
    if DEVELOPMENT_MODE:
        import traceback
        status_output.append_stdout("\n" + "".join(traceback.format_exc()) + "\n")


async def _setup_async():
    _set_busy(True)
    if not DEVELOPMENT_MODE:
        status_output.clear_output()
    status_output.append_stdout("Setting up deck...\n")
    try:
        apply_ui_settings(_collect_settings())
        result = await setup_deck()
        status_output.append_stdout(str(result) + "\n")
        deck_summary.value = str(result)
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


async def _run_async():
    _set_busy(True)
    run_state_label.value = "<b>Status:</b> SEC is running..."
    if not DEVELOPMENT_MODE:
        status_output.clear_output()
    status_output.append_stdout("Running SEC...\n")
    status_output.append_stdout(time.ctime() + "\n")
    try:
        apply_ui_settings(_collect_settings())
        settings = _collect_settings()
        void_liquid = max(0.0, float(settings["void_uL"]) - float(settings["sample_uL"])) if settings["do_void_elution"] else 0.0
        lines = [
            "Run summary:",
            f" - Column preset: {column_field.value}",
            f" - Samples: {settings['number_of_samples']} starting at {settings['first_well']}",
            f" - Swell {settings['swell_uL']:.0f} uL, sample {settings['sample_uL']:.0f} uL, elution {settings['elution_uL']:.0f} uL",
            f" - Void {settings['void_uL']:.0f} uL, void elution liquid {void_liquid:.0f} uL",
            f" - Correction added to each SEC dispense: {settings['correction_uL']:.0f} uL",
            f" - Column flow {settings['column_flow']} uL/s, into-tip dispense {settings['transfer_flow']} uL/s",
            f" - Load depth {settings['load_depth_mm']} mm below the SEC opening",
            f" - Tip rail {settings['tip_carrier_rail']}: SEC {settings['sec_tip_pos']}, sample tips {settings['sample_tip_pos']}, buffer tips {settings['buffer_tip_pos']}",
            f" - Plate rail {settings['plate_carrier_rail']}: swell {settings['buffer_plate_pos']}, sample {settings['sample_plate_pos']}, waste {settings['waste_plate_pos']}, elution {settings['elution_plate_pos']}",
            " - Piercing tool steps: skipped",
        ]
        status_output.append_stdout("\n".join(lines) + "\n\n")
        await run_sec()
        status_output.append_stdout("Run complete.\n")
        status_output.append_stdout(time.ctime() + "\n")
    except Exception as exc:
        _show_error(exc)
    finally:
        run_state_label.value = "<b>Status:</b> Idle"
        _set_busy(False)


async def _teach_async():
    _set_busy(True)
    try:
        apply_ui_settings(_collect_settings())
        await teach_load_height()
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


async def _return_teach_async():
    _set_busy(True)
    try:
        await return_teach_tip()
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


async def _return_8_async():
    _set_busy(True)
    try:
        if not deck_is_ready or lh is None:
            raise RuntimeError("Deck is not ready. Click Setup Deck first.")
        await lh.return_tips(allow_nonzero_volume=True)
        status_output.append_stdout("Returned 8-head tips.\n")
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


async def _trash_8_async():
    _set_busy(True)
    try:
        if not deck_is_ready or lh is None:
            raise RuntimeError("Deck is not ready. Click Setup Deck first.")
        await lh.discard_tips()
        status_output.append_stdout("Trashed 8-head tips.\n")
    except Exception as exc:
        _show_error(exc)
    finally:
        _set_busy(False)


setup_button.on_click(lambda _: asyncio.create_task(_setup_async()))
teach_button.on_click(lambda _: asyncio.create_task(_teach_async()))
return_teach_button.on_click(lambda _: asyncio.create_task(_return_teach_async()))
run_button.on_click(lambda _: asyncio.create_task(_run_async()))
return_8_button.on_click(lambda _: asyncio.create_task(_return_8_async()))
trash_8_button.on_click(lambda _: asyncio.create_task(_trash_8_async()))

advanced = widgets.Accordion(
    children=[
        widgets.VBox([
            column_flow_field,
            transfer_flow_field,
            aspirate_flow_field,
            swell_hold_field,
            flow_hold_field,
            drop_hold_field,
            touch_hold_field,
            drop_field,
            touch_air_field,
            margin_field,
        ]),
        widgets.VBox([
            load_depth_field,
            buffer_height_field,
            sample_height_field,
            flow_height_field,
        ]),
        widgets.VBox([tip_rail_field, plate_rail_field]),
    ]
)
advanced.set_title(0, "Flows and waits")
advanced.set_title(1, "Heights")
advanced.set_title(2, "Rails")

controls = widgets.VBox(
    [
        title,
        instructions,
        widgets.HTML("<b>Column and samples</b>"),
        column_field,
        samples_field,
        first_well_field,
        swell_field,
        void_field,
        sample_field,
        elution_field,
        correction_field,
        void_toggle,
        touch_toggle,
        development_toggle,
        widgets.HTML(
            "Correction is extra microlitres the SEC tip dispenses beyond the liquid volume. "
            "DPX says this is usually needed to finish pushing liquid through the bed. Start at 0 and increase it on a dye test."
        ),
        widgets.HTML("<b>Carrier slots</b> (1 = front, 5 = back)"),
        widgets.HTML("Tip carrier <code>TIP_CAR_480_A00</code>"),
        sec_tip_pos_field,
        sample_tip_pos_field,
        buffer_tip_pos_field,
        widgets.HTML("Plate carrier <code>PLT_CAR_L5AC_A00</code> (Hamilton 182090). Swell plate is a Corning 2 mL deep well."),
        buffer_plate_pos_field,
        sample_plate_pos_field,
        waste_plate_pos_field,
        elution_plate_pos_field,
        widgets.HTML("<b>Advanced</b>"),
        advanced,
        widgets.HBox([setup_button, teach_button, return_teach_button, run_button]),
        widgets.HTML("<b>Tip recovery</b>"),
        widgets.HBox([return_8_button, trash_8_button]),
        deck_summary,
        run_state_label,
        status_output,
    ]
)
display(controls)
